# INSTRUCTOR SOLUTION: AI Model Monitoring with Drift Detection
## AIAT 125 — Unit 5: Monitoring and Maintaining Deployed AI Models

**⚠️ INSTRUCTOR USE ONLY — Do not distribute to students**

| Task | Points |
|---|---|
| Task 1: Record baseline metrics | 20 |
| Task 2: Simulate and extend drift | 25 |
| Task 3: KS test across all features | 25 |
| Task 4: `MonitoringSystem` class | 30 |

In [ ]:
import time
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.stats import ks_2samp
import warnings
warnings.filterwarnings("ignore")

# Fixed dataset — do not change random_state
X, y = make_classification(
    n_samples=5000,
    n_features=6,
    n_informative=4,
    random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train baseline model
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")
print(f"Features         : {X.shape[1]}")
print("Setup complete — model trained.")

---
## Task 1 — Record Baseline Metrics (20 points)

**Expected output:**
```
=== Baseline Model Metrics ===
Accuracy : 0.9307
Precision: 0.9380
Recall   : 0.9231
F1-Score : 0.9305
Latency  : ~0.076 s
```

In [ ]:
start_time = time.time()
baseline_predictions = model.predict(X_test)
baseline_latency = time.time() - start_time

# SOLUTION: compute the four metrics using sklearn.metrics functions
baseline_accuracy  = accuracy_score(y_test, baseline_predictions)
baseline_precision = precision_score(y_test, baseline_predictions)
baseline_recall    = recall_score(y_test, baseline_predictions)
baseline_f1        = f1_score(y_test, baseline_predictions)

# SOLUTION: print formatted baseline report
print("=== Baseline Model Metrics ===")
print(f"Accuracy : {baseline_accuracy:.4f}")
print(f"Precision: {baseline_precision:.4f}")
print(f"Recall   : {baseline_recall:.4f}")
print(f"F1-Score : {baseline_f1:.4f}")
print(f"Latency  : {baseline_latency:.3f} s")

# Validation
assert baseline_accuracy  is not None, "Compute baseline_accuracy"
assert baseline_precision is not None, "Compute baseline_precision"
assert baseline_recall    is not None, "Compute baseline_recall"
assert baseline_f1        is not None, "Compute baseline_f1"
assert baseline_accuracy  > 0.90, f"Expected >90%, got {baseline_accuracy:.2%}"
print("Task 1 PASSED")

---
## Task 2 — Simulate Production Data Drift (25 points)

Drift types:
- **Data drift**: Input feature distributions shift
- **Concept drift**: The relationship between X and y changes

In [ ]:
# Start from a copy of the test data
X_production = X_test.copy()

# Official lab drift (provided — do not remove)
X_production[:, 0] = X_production[:, 0] + 2   # data drift on feature 0

# SOLUTION: Add a second data drift on feature 2 (shift by +1.5)
X_production[:, 2] = X_production[:, 2] + 1.5

# Concept drift: target relationship changes
y_production = np.where(X_production[:, 1] > 0, 1, 0)

# SOLUTION: Print the mean of feature 0 and feature 2 before and after drift
print(f"Feature 0: before={X_test[:, 0].mean():.2f}, after={X_production[:, 0].mean():.2f}")
print(f"Feature 2: before={X_test[:, 2].mean():.2f}, after={X_production[:, 2].mean():.2f}")

# Concept drift causes accuracy to drop because the mapping from features to labels
# has changed — the model learned the original relationship, not the new one.

# Validation
assert not np.array_equal(X_production, X_test), "X_production must differ from X_test"
assert X_production[:, 0].mean() > X_test[:, 0].mean(), "Feature 0 should have shifted up"
assert X_production[:, 2].mean() > X_test[:, 2].mean(), "Feature 2 should have shifted up"
print("Task 2 PASSED")

---
## Task 3 — KS Test Across All Features (25 points)

The KS two-sample test: p < 0.05 means the distributions are significantly different (drift detected).

In [ ]:
n_features = X_test.shape[1]

# SOLUTION: Loop through all features and run KS test
drift_results = []

for i in range(n_features):
    ks_stat, p_value = ks_2samp(X_test[:, i], X_production[:, i])
    drift = p_value < 0.05
    drift_results.append({
        "feature": i,
        "ks_stat": ks_stat,
        "p_value": p_value,
        "drift_detected": drift
    })
    print(f"Feature {i}: KS={ks_stat:.4f}, p={p_value:.2e}, {'DRIFT' if drift else 'stable'}")

# Summary
drifted_features = [r["feature"] for r in drift_results if r["drift_detected"]]
print(f"\nFeatures with drift: {drifted_features} / {n_features} total")

# Validation
assert len(drift_results) == n_features, f"Expected {n_features} results, got {len(drift_results)}"
assert any(r["drift_detected"] for r in drift_results), "At least features 0 and 2 should show drift"
assert drift_results[0]["drift_detected"], "Feature 0 must show drift (shifted by +2)"
assert drift_results[2]["drift_detected"], "Feature 2 must show drift (shifted by +1.5)"
print("Task 3 PASSED")

---
## Task 4 — `MonitoringSystem` Class (30 points)

In [ ]:
class MonitoringSystem:
    """Encapsulates all production monitoring checks for a deployed model."""

    def __init__(self):
        self.baseline = {}
        self.production = {}
        self.drift_report = {}

    def record_baseline(self, model, X_test, y_test):
        """Measure and store baseline metrics from the test set."""
        start = time.time()
        preds = model.predict(X_test)
        latency = time.time() - start

        self.baseline = {
            "accuracy":  accuracy_score(y_test, preds),
            "precision": precision_score(y_test, preds),
            "recall":    recall_score(y_test, preds),
            "f1":        f1_score(y_test, preds),
            "latency_s": latency,
        }

    def check_drift(self, X_reference, X_production, threshold=0.05):
        """Run KS test on every feature. Store results in self.drift_report.
        Return True if ANY feature shows drift.
        """
        self.drift_report = {}
        for i in range(X_reference.shape[1]):
            ks_stat, p_value = ks_2samp(X_reference[:, i], X_production[:, i])
            self.drift_report[i] = {
                "ks_stat": ks_stat,
                "p_value": p_value,
                "drift_detected": p_value < threshold,
            }
        return any(v["drift_detected"] for v in self.drift_report.values())

    def check_performance(self, model, X_production, y_production, threshold=0.1):
        """Compare production accuracy against baseline. Store in self.production.
        Return True if performance degraded.
        """
        start = time.time()
        preds = model.predict(X_production)
        latency = time.time() - start

        prod_accuracy = accuracy_score(y_production, preds)
        degraded = prod_accuracy < (self.baseline.get("accuracy", 1.0) - threshold)

        self.production = {
            "accuracy":  prod_accuracy,
            "latency_s": latency,
            "degraded":  degraded,
        }
        return degraded

    def generate_report(self):
        """Return a structured dict summarising all monitoring results."""
        baseline_acc   = self.baseline.get("accuracy", 0.0)
        production_acc = self.production.get("accuracy", 0.0)
        drifted = [k for k, v in self.drift_report.items() if v["drift_detected"]]

        return {
            "baseline_accuracy":    round(baseline_acc, 4),
            "production_accuracy":  round(production_acc, 4),
            "accuracy_drop":        round(baseline_acc - production_acc, 4),
            "data_drift_detected":  len(drifted) > 0,
            "drifted_features":     drifted,
            "performance_degraded": self.production.get("degraded", False),
        }


# --- Run the monitoring system ---
monitor = MonitoringSystem()
monitor.record_baseline(model, X_test, y_test)
monitor.check_drift(X_test, X_production)
monitor.check_performance(model, X_production, y_production)
report = monitor.generate_report()

print("=== Monitoring Summary Report ===")
for key, value in report.items():
    print(f"  {key}: {value}")

In [ ]:
# --- Final validation ---

assert isinstance(report, dict), "generate_report() must return a dict"

required_keys = ["baseline_accuracy", "production_accuracy", "accuracy_drop",
                 "data_drift_detected", "drifted_features", "performance_degraded"]
for k in required_keys:
    assert k in report, f"Report missing key: '{k}'"

assert report["baseline_accuracy"] > 0.90, "Baseline accuracy should be > 90%"
assert report["production_accuracy"] < 0.60, "Production accuracy should drop significantly"
assert report["data_drift_detected"] == True, "Drift should be detected"
assert report["performance_degraded"] == True, "Degradation should be detected"
assert len(report["drifted_features"]) >= 2, "At least features 0 and 2 should drift"

print("=== ALL TASKS PASSED ===")
print(f"Baseline accuracy    : {report['baseline_accuracy']:.2%}")
print(f"Production accuracy  : {report['production_accuracy']:.2%}")
print(f"Accuracy drop        : {report['accuracy_drop']:.2%}")
print(f"Data drift detected  : {report['data_drift_detected']}")
print(f"Drifted features     : {report['drifted_features']}")
print(f"Performance degraded : {report['performance_degraded']}")